# CFD Geometry — local quick start (VS Code)

Run this notebook after cloning the repository and creating a virtual environment.

| Step | Task |
|------|------|
| 1 | Editable install from repo |
| 2 | Draw study area on map |
| 3 | Download OSM and build STLs |
| 4 | List outputs and 3D preview |

### Setup (once per machine)

1. **File → Open Folder** → CFDGeometry repo root (`src/`, `pyproject.toml`).
2. Create venv and install:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip setuptools wheel
python -m pip install -e ".[notebook,download]"
```

3. Open this notebook and select the **`.venv`** kernel.


## 1. Verify install

Re-run after `git pull` if imports fail.


In [ ]:
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "src" / "cfd_geometry" / "domain").is_dir():
            return parent
    return here


repo = find_repo_root()
if not (repo / "src" / "cfd_geometry" / "domain").is_dir():
    raise RuntimeError(
        "Open the CFDGeometry repo root in VS Code (folder containing src/), "
        "then select the .venv kernel."
    )

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo}[notebook,download]"]
)

import cfd_geometry
from cfd_geometry.domain import DomainConfig, build_domain  # noqa: F401

print("Repo:", repo)
print("cfd_geometry", cfd_geometry.__version__)


## 2. Define study area

Draw a rectangle on the map, then click **Use this extent**.


In [ ]:
from cfd_geometry.notebook import select_extent

PLACE = "Milwaukee, Wisconsin, USA"
CENTER = None

selector = select_extent(place=PLACE if CENTER is None else None, center=CENTER)
selector


## 3. Confirm extent


In [ ]:
if selector.bbox is None:
    raise RuntimeError("Draw a rectangle, then click 'Use this extent'.")

bbox = selector.bbox
print(f"west={bbox.west:.6f}  south={bbox.south:.6f}")
print(f"east={bbox.east:.6f}  north={bbox.north:.6f}")


## 4. Build geometry


In [ ]:
from pathlib import Path

from cfd_geometry.domain import DomainConfig, build_domain

result = build_domain(
    DomainConfig(
        output_dir=Path("data"),
        bbox=selector.bbox,
        run_download=True,
        download_layers=("buildings", "trees"),
        build_buildings=True,
        build_trees=True,
        height_source="composite",
    )
)
result.stl_files


## 5. Output files


In [ ]:
from pathlib import Path

for path in sorted(Path("data/output").glob("*.stl")):
    print(f"{path.name:24s} {path.stat().st_size / 1024:8.1f} KiB")


## 6. 3D preview


In [ ]:
from cfd_geometry.notebook.visualize import plot_domain_stls

plot_domain_stls(result, layers=("buildings", "trees"), max_triangles=8000)
